- Iterate over all subjects in Exported
- Load magnitude_enc0.nii and mask.nii
- Split the 4D volumes into individual cardiac frames
- Save each frame as an independent nnU-Net case

- Preserve the header (not only the affine).
- Copy the header before saving each frame.
- Validate that mask and magnitude shapes match.

In [1]:
from pathlib import Path
import re
import nibabel as nib
import numpy as np

In [31]:
EXPORT_DIR = Path("D:/projects/4D-flow-MRI-aortic-segmentation-/scripts/Exported")
OUTPUT_DIR = Path("train_data")
IMG_OUT = Path("train_data/ao_imgs")
SEG_OUT = Path("train_data/ao_segs")
IMG_OUT.mkdir(parents=True, exist_ok=True)
SEG_OUT.mkdir(parents=True, exist_ok=True)

In [32]:
print(IMG_OUT)
print(SEG_OUT)

train_data\ao_imgs
train_data\ao_segs


In [33]:
print(f"Exported directory: {EXPORT_DIR}")

Exported directory: D:\projects\4D-flow-MRI-aortic-segmentation-\scripts\Exported


In [34]:
subjects = sorted(EXPORT_DIR.glob("VirtualSubject_n*"))

In [35]:
total_cases = 0

# ---------------------------------------------------
# Loop subjects
# ---------------------------------------------------

for subject_dir in sorted(EXPORT_DIR.glob("VirtualSubject_n*")):

    mri_dir = subject_dir / "mri_gzip"

    mag_file = mri_dir / "magnitude_enc0.nii.gz"
    mask_file = mri_dir / "mask.nii.gz"

    if not mag_file.exists():
        print(f"Missing: {mag_file}")
        continue

    if not mask_file.exists():
        print(f"Missing: {mask_file}")
        continue

    subject_match = re.search(r"n(\d+)", subject_dir.name)
    subject_id = int(subject_match.group(1))

    print(f"Processing {subject_dir.name}")

    mag_img = nib.load(str(mag_file))
    mask_img = nib.load(str(mask_file))

    mag = mag_img.get_fdata()
    mask = mask_img.get_fdata()

    print("magnitude Shape:", mag.shape)
    print("mask Shape:", mask.shape)

    if mag.shape != mask.shape:
        raise RuntimeError(
            f"{subject_dir.name}: magnitude and mask shapes differ: "
            f"{mag.shape} vs {mask.shape}"
        )

    if mag.ndim != 4:
        raise RuntimeError(
            f"{subject_dir.name}: expected 4D volume (X,Y,Z,T), got {mag.shape}"
        )

    n_frames = mag.shape[-1]

    for frame in range(n_frames):

        img_frame = mag[..., frame].astype(np.float32)

        # mask is already binary [0,1]
        seg_frame = mask[..., frame].astype(np.uint8)

        case_id = f"VS{subject_id:03d}_f{frame:02d}"

        img_out = IMG_OUT / f"{case_id}_0000.nii.gz"
        seg_out = SEG_OUT / f"{case_id}.nii.gz"

        # preserve metadata
        img_header = mag_img.header.copy()
        seg_header = mask_img.header.copy()

        nib.save(
            nib.Nifti1Image(
                img_frame,
                affine=mag_img.affine,
                header=img_header
            ),
            str(img_out)
        )

        nib.save(
            nib.Nifti1Image(
                seg_frame,
                affine=mask_img.affine,
                header=seg_header
            ),
            str(seg_out)
        )

        total_cases += 1

print()
print("=" * 60)
print(f"Created {total_cases} nnU-Net training samples")
print("=" * 60)

Processing VirtualSubject_n001
magnitude Shape: (115, 100, 21, 25)
mask Shape: (115, 100, 21, 25)
Processing VirtualSubject_n002
magnitude Shape: (123, 116, 26, 25)
mask Shape: (123, 116, 26, 25)
Processing VirtualSubject_n003
magnitude Shape: (159, 180, 29, 25)
mask Shape: (159, 180, 29, 25)
Processing VirtualSubject_n004
magnitude Shape: (126, 144, 27, 25)
mask Shape: (126, 144, 27, 25)
Processing VirtualSubject_n005
magnitude Shape: (140, 112, 32, 25)
mask Shape: (140, 112, 32, 25)
Processing VirtualSubject_n006
magnitude Shape: (127, 156, 32, 25)
mask Shape: (127, 156, 32, 25)
Processing VirtualSubject_n007
magnitude Shape: (113, 100, 24, 25)
mask Shape: (113, 100, 24, 25)
Processing VirtualSubject_n008
magnitude Shape: (112, 84, 20, 25)
mask Shape: (112, 84, 20, 25)
Processing VirtualSubject_n009
magnitude Shape: (98, 156, 30, 25)
mask Shape: (98, 156, 30, 25)
Processing VirtualSubject_n010
magnitude Shape: (140, 108, 27, 25)
mask Shape: (140, 108, 27, 25)
Processing VirtualSubjec

81 subjects × 25 frames = 2025 samples

### test cases:

In [15]:
EXPORT_DIR = Path("/workspace/scripts/Exported")
OUTPUT_DIR = Path("test_data")
IMG_OUT = Path("test_data/ao_imgs")
SEG_OUT = Path("test_data/ao_segs")
IMG_OUT.mkdir(parents=True, exist_ok=True)
SEG_OUT.mkdir(parents=True, exist_ok=True)

In [16]:
subjects = sorted(EXPORT_DIR.glob("VirtualSubject_n*"))

In [17]:
subjects

[PosixPath('/workspace/scripts/Exported/VirtualSubject_n001'),
 PosixPath('/workspace/scripts/Exported/VirtualSubject_n002'),
 PosixPath('/workspace/scripts/Exported/VirtualSubject_n003'),
 PosixPath('/workspace/scripts/Exported/VirtualSubject_n004'),
 PosixPath('/workspace/scripts/Exported/VirtualSubject_n005'),
 PosixPath('/workspace/scripts/Exported/VirtualSubject_n006'),
 PosixPath('/workspace/scripts/Exported/VirtualSubject_n007'),
 PosixPath('/workspace/scripts/Exported/VirtualSubject_n008'),
 PosixPath('/workspace/scripts/Exported/VirtualSubject_n009'),
 PosixPath('/workspace/scripts/Exported/VirtualSubject_n010'),
 PosixPath('/workspace/scripts/Exported/VirtualSubject_n011'),
 PosixPath('/workspace/scripts/Exported/VirtualSubject_n012'),
 PosixPath('/workspace/scripts/Exported/VirtualSubject_n013'),
 PosixPath('/workspace/scripts/Exported/VirtualSubject_n014'),
 PosixPath('/workspace/scripts/Exported/VirtualSubject_n015'),
 PosixPath('/workspace/scripts/Exported/VirtualSubject_

In [18]:
total_cases = 0

# ---------------------------------------------------
# Loop subjects
# ---------------------------------------------------

for subject_dir in sorted(EXPORT_DIR.glob("VirtualSubject_n*")):

    mri_dir = subject_dir / "mri_gzip"

    mag_file = mri_dir / "magnitude_enc0.nii.gz"
    mask_file = mri_dir / "mask.nii.gz"

    if not mag_file.exists():
        print(f"Missing: {mag_file}")
        continue

    if not mask_file.exists():
        print(f"Missing: {mask_file}")
        continue

    subject_match = re.search(r"n(\d+)", subject_dir.name)
    subject_id = int(subject_match.group(1))

    print(f"Processing {subject_dir.name}")

    mag_img = nib.load(str(mag_file))
    mask_img = nib.load(str(mask_file))

    mag = mag_img.get_fdata()
    mask = mask_img.get_fdata()

    print("magnitude Shape:", mag.shape)
    print("mask Shape:", mask.shape)

    if mag.shape != mask.shape:
        raise RuntimeError(
            f"{subject_dir.name}: magnitude and mask shapes differ: "
            f"{mag.shape} vs {mask.shape}"
        )

    if mag.ndim != 4:
        raise RuntimeError(
            f"{subject_dir.name}: expected 4D volume (X,Y,Z,T), got {mag.shape}"
        )

    n_frames = mag.shape[-1]

    for frame in range(n_frames):

        img_frame = mag[..., frame].astype(np.float32)

        # mask is already binary [0,1]
        seg_frame = mask[..., frame].astype(np.uint8)

        case_id = f"VS{subject_id:03d}_f{frame:02d}"

        img_out = IMG_OUT / f"{case_id}_0000.nii.gz"
        seg_out = SEG_OUT / f"{case_id}.nii.gz"

        # preserve metadata
        img_header = mag_img.header.copy()
        seg_header = mask_img.header.copy()

        nib.save(
            nib.Nifti1Image(
                img_frame,
                affine=mag_img.affine,
                header=img_header
            ),
            str(img_out)
        )

        nib.save(
            nib.Nifti1Image(
                seg_frame,
                affine=mask_img.affine,
                header=seg_header
            ),
            str(seg_out)
        )

        total_cases += 1

print()
print("=" * 60)
print(f"Created {total_cases} nnU-Net test samples")
print("=" * 60)

Processing VirtualSubject_n001
magnitude Shape: (115, 100, 21, 25)
mask Shape: (115, 100, 21, 25)
Processing VirtualSubject_n002
magnitude Shape: (123, 116, 26, 25)
mask Shape: (123, 116, 26, 25)
Processing VirtualSubject_n003
magnitude Shape: (159, 180, 29, 25)
mask Shape: (159, 180, 29, 25)
Processing VirtualSubject_n004
magnitude Shape: (126, 144, 27, 25)
mask Shape: (126, 144, 27, 25)
Processing VirtualSubject_n005
magnitude Shape: (140, 112, 32, 25)
mask Shape: (140, 112, 32, 25)
Processing VirtualSubject_n006
magnitude Shape: (127, 156, 32, 25)
mask Shape: (127, 156, 32, 25)
Processing VirtualSubject_n007
magnitude Shape: (113, 100, 24, 25)
mask Shape: (113, 100, 24, 25)
Processing VirtualSubject_n008
magnitude Shape: (112, 84, 20, 25)
mask Shape: (112, 84, 20, 25)
Processing VirtualSubject_n009
magnitude Shape: (98, 156, 30, 25)
mask Shape: (98, 156, 30, 25)
Processing VirtualSubject_n010
magnitude Shape: (140, 108, 27, 25)
mask Shape: (140, 108, 27, 25)
Processing VirtualSubjec